# Baseline Modeling: Predicting Quiz Performance from Session Engagement

This notebook trains baseline models (Linear Regression, Random Forest) using the reusable modeling utilities in `models/baseline_models.py`.

We will:
- Load the cleaned session-level dataset
- Define numeric and categorical features
- Build preprocessing pipeline
- Train/test split (stratified by condition)
- Train Linear Regression
- Train Random Forest
- Compare performance


TODO: Organize the notebook
1. Introduction / Problem framing
What are we predicting?
Why does it matter?
What’s the dataset?
What’s the modeling goal?
2. Data loading + quick sanity checks
.head(), .info(), .describe()
Missingness overview
Basic distributions
3. Feature engineering (your behavioral metrics)
You already did the heavy lifting here — now you just narrate it:
Why avg/median/std response time?
Why rapid response %?
Why session-level features?
This is where you shine because you can explain the intuition behind each feature.
4. Preprocessing pipeline
Show:
numeric imputer
scaler
categorical encoder
ColumnTransformer
This is your “look how clean and reproducible this is” moment.
5. Modeling
Baseline LinearRegression (weak, but important as a reference)
RandomForest (your strong performer)
Maybe XGBoost if you want to flex
6. Evaluation
MAE, RMSE, R²
Predictions vs actuals
Residual plot (optional)
7. Feature importances
This is where the story emerges:
Which behaviors predict quiz performance?
Are fast responders doing better?
Does session type matter?
Are there condition effects?
8. Interpretation + limitations
Human behavior is noisy
Quiz performance is multi-factor
Nonlinear models capture structure better
Future work (per-condition models, SHAP, etc.)

In [2]:
import pandas as pd

from models.baseline_models import (
    build_preprocessor,
    build_linear_model,
    build_random_forest,
    build_xgboost_model,
    split_data,
    evaluate_model
)
data_dir = '../data/'

In [3]:
data = pd.read_csv(data_dir + "sessions_with_engagement_features_updated.csv")
data.head()
data.info()
data.describe(include="all")
data.columns
df = data[data['has_both'] == True]

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 115 entries, 0 to 114
Data columns (total 25 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   user_id                   115 non-null    object 
 1   session_type              115 non-null    object 
 2   status                    115 non-null    object 
 3   condition                 115 non-null    float64
 4   start_time                115 non-null    float64
 5   end_time                  115 non-null    float64
 6   duration_seconds          115 non-null    float64
 7   total_messages            115 non-null    float64
 8   user_messages             115 non-null    float64
 9   assistant_messages        115 non-null    float64
 10  quiz_score                115 non-null    float64
 11  quiz_total                115 non-null    float64
 12  quiz_percentage           115 non-null    float64
 13  quiz_completed_time       115 non-null    float64
 14  survey_com

In [4]:
numeric_features = [
    "duration_seconds",
    "total_messages",
    "user_messages",
    "assistant_messages",
    "avg_response_time",
    "median_response_time",
    "std_response_time",
    "min_response_time",
    "max_response_time",
    "rapid_response_count",
    "rapid_response_pct",
    "avg_difficulty_correct",
]

categorical_features = [
    "condition",
    "session_type",
    "has_both"
]
target = "quiz_percentage"


In [5]:
preprocessor = build_preprocessor(numeric_features, categorical_features)
preprocessor

,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,missing_values,nan
,strategy,'median'
,fill_value,None


In [6]:
df[categorical_features].isna().sum()


condition       0
session_type    0
has_both        0
dtype: int64

In [7]:
X_train, X_test, y_train, y_test = split_data(
    df,
    target="quiz_percentage",
    stratify_col="condition"
)


In [8]:
[col for col in numeric_features if col not in df.columns]
[col for col in categorical_features if col not in df.columns]
[col for col in df.columns if "response" in col.lower()]
preprocessor

,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,missing_values,nan
,strategy,'median'
,fill_value,None


In [9]:
lin_model = build_linear_model(preprocessor)
lin_model.fit(X_train, y_train)
# For linear regression, get coefficients
feature_names = lin_model.named_steps["preprocessor"].get_feature_names_out()
coefs = lin_model.named_steps["regressor"].coef_

for name, coef in zip(feature_names, coefs):
    print(f"{name}: {coef:.4f}")


num__duration_seconds: 15.7995
num__total_messages: -2.8503
num__user_messages: 2.3668
num__assistant_messages: -8.3548
num__avg_response_time: -32.7254
num__median_response_time: 15.6943
num__std_response_time: 11.3382
num__min_response_time: 8.3286
num__max_response_time: -2.1968
num__rapid_response_count: 4.0603
num__rapid_response_pct: -2.6128
num__avg_difficulty_correct: -1.4195
cat__condition_1.0: 6.5193
cat__condition_2.0: 15.8610
cat__condition_3.0: -22.3803
cat__session_type_arraylist: -6.6992
cat__session_type_recursion: 6.6992
cat__has_both_True: 0.0000


In [10]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

y_pred = lin_model.predict(X_test)

print("MAE:", mean_absolute_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print("R²:", r2_score(y_test, y_pred))


MAE: 18.206058931376795
RMSE: 28.140098340653285
R²: -0.22962156467655714


In [11]:
pd.DataFrame({
    "actual": y_test.values,
    "predicted": y_pred
}).head(10)


,actual,predicted
0,80.0,83.554314
1,40.0,50.914952
2,80.0,71.479515
3,80.0,86.240678
4,40.0,63.505193
5,80.0,74.357868
6,100.0,91.284527
7,40.0,120.737604
8,100.0,68.374100
9,60.0,59.486733


In [12]:
rf_model = build_random_forest(preprocessor, n_estimators=300, max_depth=None)

rf_model.fit(X_train, y_train)

y_pred = rf_model.predict(X_test)

print("MAE:", mean_absolute_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print("R²:", r2_score(y_test, y_pred))
rf_model

MAE: 11.555555555555555
RMSE: 19.38283755115312
R²: 0.41661539906103295


,steps,"[('preprocessor', ...), ('regressor', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [13]:
importances = rf_model.named_steps["regressor"].feature_importances_
feature_names = rf_model.named_steps["preprocess"].get_feature_names_out()

sorted(list(zip(feature_names, importances)), key=lambda x: x[1], reverse=True)[:15]


KeyError: 'preprocess'

XGBoost

In [3]:
!pip install xgboost


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [14]:

xgb_model = build_xgboost_model(preprocessor)
xgb_model.fit(X_train, y_train)

y_pred = xgb_model.predict(X_test)

print("MAE:", mean_absolute_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))
print("R²:", r2_score(y_test, y_pred))


MAE: 12.280542282831101
RMSE: 19.589929435864843
R²: 0.40408270116811895
